This notebook gives an Exploratory Data Analysis of the ASL dataset.

The *GOAL* of us as follows:
1) Data Cleaning

2) Descriptive Statistics

3) Data Visualization

4) Feature Engineering
Preprocessing notebook coming soon.

# Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from tqdm.notebook import tqdm
import os
import math
import json
import re
import tensorflow as tf

# TQDM Progress Bar With Pandas Apply Function
tqdm.pandas()

# Data

Character To Ordinal Encoding

In [ ]:
# Read Character to Ordinal Encoding Mapping
with open('/kaggle/input/asl-fingerspelling/character_to_prediction_index.json') as json_file:
    CHAR2ORD = json.load(json_file)

In [ ]:
# Read Train DataFrame
df = pd.read_csv('/kaggle/input/asl-fingerspelling/train.csv')
# Number Of Train Samples
N_SAMPLES = len(df)
print(f'N_SAMPLES: {N_SAMPLES}')
display(df.head())

In [ ]:
# Read First Parquet File
parquet_df = pd.read_parquet(str('/kaggle/input/asl-fingerspelling/')+df['path'][0])

# Each parquet file contains 1000 recordings
print(f'# Unique Recording: {parquet_df.index.nunique()}')
# Display DataFrame layout
display(parquet_df.head())

## Feature Engineering

In [ ]:
"""
Attempt to retrieve phrase type
Could be used for pretraining or type specific inference
 *) Phone Number\
 *) URL
 *3) Addres
"""
def get_phrase_type(phrase):
    # Phone Number
    if re.match(r'^[\d+-]+$', phrase):
        return 'phone_number'
    # url
    elif any([substr in phrase for substr in ['www', '.', '/']]) and ' ' not in phrase:
        return 'url'
    # Address
    else:
        return 'address'
    
df['phrase_type'] = df['phrase'].apply(get_phrase_type)

# Get complete file path to file
def get_file_path(path):
    return f'/kaggle/input/asl-fingerspelling/{path}'
df['file_path'] = df['path'].apply(get_file_path)

# Get phrase character to Character Tuple
def get_char_tuple(phrase):
    return tuple(phrase)
df['phrase_char'] = df['phrase'].apply(get_char_tuple)

# Get Character Tuple Length
def get_char_tuple_len(c):
    return len(c)
df['phrase_char_len'] = df['phrase_char'].apply(get_char_tuple_len)

# Train DataFrame indexed by sequence_id to convenientlyy lookup recording data
df_sequence_id = df.set_index('sequence_id')

## Descriptive Statistic

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Number of parquet chunks to analyse
IS_INTERACTIVE = os.environ['KAGGLE_KERNEL_RUN_TYPE'] == 'Interactive'
SEED = 42

global parquet_df

N = 5 if IS_INTERACTIVE else 25
# Number of Unique Frames in Recording
N_UNIQUE_FRAMES = []

UNIQUE_FILE_PATHS = pd.Series(df['file_path'].unique())

for idx, file_path in enumerate(tqdm(UNIQUE_FILE_PATHS.sample(N, random_state=SEED))):
    parquet_df = pd.read_parquet(file_path)
    for group, group_df in parquet_df.groupby('sequence_id'):
        N_UNIQUE_FRAMES.append(group_df['frame'].nunique())

# Convert to Numpy Array
N_UNIQUE_FRAMES = np.array(N_UNIQUE_FRAMES)

# Number of unique frames in each video
PERCENTILES = [0.01, 0.10, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
display(pd.Series(N_UNIQUE_FRAMES).describe(percentiles=PERCENTILES).to_frame('Value').astype(int))

## Data Visualization

In [ ]:
# Number of unique characters
UNIQUE_CHARACTERS = np.unique(np.concatenate(df['phrase_char']))
N_UNIQUE_CHARACTERS = len(UNIQUE_CHARACTERS)
print(f'N_UNIQUE_CHARACTERS: {N_UNIQUE_CHARACTERS}')

In [ ]:
# Count the number of occurrences of each fruit
counts = df['phrase_type'].value_counts()

# Create a pie chart using plotly.express
fig = px.pie(counts, values=counts.values, names=counts.index)

# Show the pie chart
fig.show()

In [ ]:
# Character Count Occurance
plt.figure(figsize=(15,8))
plt.title('Character Length Occurance of Phrases')
df['phrase_char_len'].value_counts().sort_index().plot(kind='bar')
plt.xlim(-0.50, df['phrase_char_len'].max() - 1.50)
plt.xlabel('Pharse Character Length')
plt.ylabel('Sample Count')
plt.grid(axis='y')
plt.show()

In [ ]:
print(f'Number of parquet files:{len(df["path"].unique())}')
print(f'Number of phrase : {len(df["phrase"].unique())}')
print(f'Number of sequence id: {len(df["sequence_id"].unique())}')

Through those numbers we can know that each parquet file contains more than a phrase. Each phrase can repeat by many time.

In [ ]:
df["participant_id"].value_counts()

In [ ]:
# Calculate the counts for each participant_id
counts = df["participant_id"].value_counts()

# Set up the figure and axes
fig, ax = plt.subplots(figsize=(15, 6))

# Plot the histogram
bars = ax.bar(counts.index, counts.values,width=5)

# Set the labels and title
ax.set_xlabel("Participant ID")
ax.set_ylabel("Total Row Count")
ax.set_title("Row Counts by Participant ID")

# Show the plot
plt.show()

In [ ]:
# Set the custom color scheme
color_scheme = ["#4f000b", "#720026", "#ce4257", "#ff7f51", "#ff9b54"]
df["participant_id"] = df["participant_id"].astype(str)
counts = df["participant_id"].value_counts()
fig, ax = plt.subplots(figsize=(12, 6))
# Plot the histogram
bars = ax.bar(counts.index, counts.values, color=color_scheme[1])
# Set the labels and title
ax.set_xlabel("Participant ID")
ax.set_ylabel("Total Row Count")
ax.set_title("Row Counts by Participant ID")
# Rotate the x-axis labels if needed
plt.xticks(rotation=90, ha='center')
plt.xlim(-1, 94)

# Show the plot
plt.show()

### Count of each character

In [ ]:
import plotly.graph_objects as go
character_count = {}
for phrase in df["phrase"]:
    for char in phrase:
        try:
            character_count[char] += 1
        except KeyError:
            character_count[char] = 1
character_count = dict(sorted(character_count.items(), key=lambda k: k[1], reverse=True))
fig = go.Figure(
    data=[
        go.Bar(
            x=list(character_count.keys()),
            y=list(character_count.values()),
            hovertemplate='Character: <b>%{x}</b><br>Count: <b>%{y}</b><extra></extra>'
        )
    ]
)
fig.update_traces(
    marker_color='rgb(158,202,225)',
    marker_line_color='rgb(8,48,107)',
    marker_line_width=1.5,
)
fig.update_layout(
    title={
        'text':f'<b>Count of each character</b>',
        'y':0.95,
        'x':0.5,
        'xanchor':'center',
        'yanchor':'top'
    },
    xaxis_title="Character",
    yaxis_title="Count",
    height=500,
    title_x=0.5,
    hoverlabel=dict(
        font_size=14,
    ),
    title_font_size=24,
    template="plotly_white",
    margin=dict(t=40,b=20,l=20,r=20),
    yaxis_tickformat='digits',
    margin_pad=5
)
fig.show()

In [ ]:
parquet_df[parquet_df.index==165414438]

In [ ]:
from plotly.subplots import make_subplots
# Loop over all parquet files and collect data
frames = {} # Save total frames as a dict
file_list = df["path"].unique() # List of files
for index, path in enumerate(file_list):
    a = parquet_df # read the corresponding parquet file into a dataframe
    frame_count = a.groupby(a.index)["frame"].count().tolist() # count the number of frames and store into an array
    frames[path.split("/")[1].split(".")[0]] = frame_count # parse the label for the file
def plot_frame_distribution(frames):
    fig = make_subplots(
        rows=1,
        cols=1,  # Số cột giảm xuống 1
        horizontal_spacing=0.01
    )

    for key in frames.keys():
        trace1 = go.Scatter(
            y=frames[key],
            name=key,
            hovertemplate='Sequence Number: <b>%{x}</b><br>Number of frames: <b>%{y}</b><extra></extra>',
            marker_color='#484bfa',
            opacity=0.75,
        )
        fig.add_trace(trace1)
    length_of_data = len(fig.data)
    length_of_frames = len(frames.keys())
    for data in range(1, length_of_data):
        fig.update_traces(visible=False, selector=data)
    fig.show()
plot_frame_distribution(frames)

In [ ]:
df['phrase_len'] = df.phrase.str.len()
sns.histplot(df.phrase_len, kde=True, binwidth = 2)
plt.title('Character occurences in each phrase in training set')
plt.xlabel('Phrase length')
plt.ylabel('Sample Count')

In [ ]:
plt.figure(figsize=(15,8))
plt.title('Number of Unique Frames', size=24)
pd.Series(N_UNIQUE_FRAMES).plot(kind='hist', bins=128)
plt.grid()
xlim = math.ceil(plt.xlim()[1])
plt.xlim(0, xlim)
plt.xticks(np.arange(0, xlim+50, 50))
plt.show()

In [ ]:
# With N_TARGET_FRAMES = 256 ~85% will be below
N_UNIQUE_FRAMES_WATERFALL = []
# Maximum Number of Unique Frames to use
N_MAX_UNIQUE_FRAMES = 400
# Compute Percentage
for n in tqdm(range(0,N_MAX_UNIQUE_FRAMES+1)):
    N_UNIQUE_FRAMES_WATERFALL.append(sum(N_UNIQUE_FRAMES >= n) / len(N_UNIQUE_FRAMES) * 100)

plt.figure(figsize=(18,10))
plt.title('Waterfall Plot For Number Of Unique Frames')
pd.Series(N_UNIQUE_FRAMES_WATERFALL).plot(kind='bar')
plt.grid(axis='y')
plt.xticks([1] + np.arange(5, N_MAX_UNIQUE_FRAMES+5, 5).tolist(), size=8, rotation=45)
plt.xlabel('Number of Unique Frames', size=16)
plt.yticks(np.arange(0, 100+5, 5), [f'{i}%' for i in range(0,100+5,5)])
plt.ylim(0, 100)
plt.ylabel('Percentage of Samples With At Least N Unique Frames', size=16)
plt.show()

## Landmark Indices

In [ ]:
def get_idxs(parquet_df, words_pos, words_neg=[], ret_names=True, idxs_pos=None):
    idxs = []
    names = []
    for w in words_pos:
        for col_idx, col in enumerate(parquet_df.columns):
            # Exclude Non Landmark Columns
            if col in ['frame']:
                continue
                
            col_idx = int(col.split('_')[-1])
            # Check if column name contains all words
            if (w in col) and (idxs_pos is None or col_idx in idxs_pos) and all([w not in col for w in words_neg]):
                idxs.append(col_idx)
                names.append(col)
    # Convert to Numpy arrays
    idxs = np.array(idxs)
    names = np.array(names)
    # Returns either both column indices and names
    if ret_names:
        return idxs, names
    # Or only columns indices
    else:
        return idxs

In [ ]:
# Lips Landmark Face Ids
LIPS_LANDMARK_IDXS = np.array([
        61, 185, 40, 39, 37, 0, 267, 269, 270, 409,
        291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
        78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
        95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
    ])

# Landmark Indices for Left/Right hand without z axis in raw data
LEFT_HAND_IDXS0, LEFT_HAND_NAMES0 = get_idxs(parquet_df, ['left_hand'], ['z'])
RIGHT_HAND_IDXS0, RIGHT_HAND_NAMES0 = get_idxs(parquet_df, ['right_hand'], ['z'])
# POSE_IDXS0, POSE_NAMES0 = get_idxs(parquet_df, ['pose'], ['z'])
LIPS_IDXS0, LIPS_NAMES0 = get_idxs(parquet_df, ['face'], ['z'], idxs_pos=LIPS_LANDMARK_IDXS)
COLUMNS0 = np.concatenate((LEFT_HAND_NAMES0, RIGHT_HAND_NAMES0, LIPS_NAMES0,#POSE_NAMES0))
N_COLS0 = len(COLUMNS0)
# Only X/Y axes are used
N_DIMS0 = 2

print(f'N_COLS0: {N_COLS0}')

In [ ]:
# Landmark Indices in subset of dataframe with only COLUMNS selected
LEFT_HAND_IDXS = np.argwhere(np.isin(COLUMNS0, LEFT_HAND_NAMES0)).squeeze()
RIGHT_HAND_IDXS = np.argwhere(np.isin(COLUMNS0, RIGHT_HAND_NAMES0)).squeeze()
LIPS_IDXS = np.argwhere(np.isin(COLUMNS0, LIPS_NAMES0)).squeeze()
N_COLS = N_COLS0
# Only X/Y axes are used
N_DIMS = 2

print(f'N_COLS: {N_COLS}')

In [ ]:
# Indices in processed data by axes with only dominant hand
HAND_X_IDXS = np.array(
        [idx for idx, name in enumerate(LEFT_HAND_NAMES0) if 'x' in name]
    ).squeeze()
HAND_Y_IDXS = np.array(
        [idx for idx, name in enumerate(LEFT_HAND_NAMES0) if 'y' in name]
    ).squeeze()
# Names in processed data by axes
HAND_X_NAMES = LEFT_HAND_NAMES0[HAND_X_IDXS]
HAND_Y_NAMES = LEFT_HAND_NAMES0[HAND_Y_IDXS]

## Data Cleaning

In [ ]:
"""
    Tensorflow layer to process data in TFLite
    Data needs to be processed in the model itself, so we can not use Python
""" 
class PreprocessLayerNonNaN(tf.keras.layers.Layer):
    def __init__(self):
        super(PreprocessLayerNonNaN, self).__init__()
    
    @tf.function(
        input_signature=(tf.TensorSpec(shape=[None,N_COLS0], dtype=tf.float32),),
    )
    def call(self, data0):
        # Fill NaN Values With 0
        data = tf.where(tf.math.is_nan(data0), 0.0, data0)
        
        # Hacky
        data = data[None]
        
        # Empty Hand Frame Filtering
        hands = tf.slice(data, [0,0,0], [-1, -1, 84])
        hands = tf.abs(hands)
        mask = tf.reduce_sum(hands, axis=2)
        mask = tf.not_equal(mask, 0)
        data = data[mask][None]
        data = tf.squeeze(data, axis=[0])
        
        return data
    
preprocess_layer_non_nan = PreprocessLayerNonNaN()

In [ ]:
# Unique Parquet Files
UNIQUE_FILE_PATHS = pd.Series(df['file_path'].unique())
# Number of parquet chunks to analyse
N = 5 if (IS_INTERACTIVE or FAST) else len(UNIQUE_FILE_PATHS)
# Number of Non Nan Frames in Recording
N_NON_NAN_FRAMES = []

for idx, file_path in enumerate(tqdm(UNIQUE_FILE_PATHS.sample(N, random_state=SEED))):
    parquet_df = pd.read_parquet(file_path)
    for group, group_df in parquet_df.groupby('sequence_id'):
        frames = preprocess_layer_non_nan(group_df[COLUMNS0].values).numpy()
        N_NON_NAN_FRAMES.append(len(frames))

# Convert to Numpy Array
N_NON_NAN_FRAMES = pd.Series(N_NON_NAN_FRAMES).to_frame('# Frames')

In [ ]:
N_NON_NAN_FRAMES.plot(kind='hist', bins=128, figsize=(15,8))
plt.title('Number of Non NaN Frames', size=24)
plt.grid()
xlim = np.percentile(N_NON_NAN_FRAMES, 99)
plt.xlim(0, xlim)
plt.xticks(np.arange(0, xlim+32, 32))
plt.show()


### % NA từng bộ phận 

In [ ]:
POSE = parquet_df.loc[:, POSE_NAMES0] 
LIPS = parquet_df.loc[:, LIPS_NAMES0] 
RIGHT_HAND = parquet_df.loc[:, RIGHT_HAND_NAMES0] 
LEFT_HAND = parquet_df.loc[:, LEFT_HAND_NAMES0] 
RIGHT_HAND

In [ ]:
print(f"Percentage of nulls in Left Hand data = {100*np.mean(LEFT_HAND['x_left_hand_0'].isnull()):.02f} %")
print(f"Percentage of nulls in Right Hand data = {100*np.mean(RIGHT_HAND['x_right_hand_0'].isnull()):.02f} %")
print(f"Percentage of nulls in Pose data = {100*np.mean(POSE['x_pose_0'].isnull()):.02f}%")
print(f"Percentage of nulls in Lips data = {100*np.mean(LIPS['x_face_0'].isnull()):.02f}%")